# 02 — Silver Layer Transformation

**Purpose:** Clean, conform, deduplicate, and MERGE Bronze data into Silver staging tables.

**What this notebook does:**
- Accepts `pipeline_run_date` — reads only the matching Bronze partition
- Resolves student identity across HubSpot, Dynamics, and Paradigm via `student_email`
- Implements SCD Type 2 on `silver.student` for `visa_code` and `is_international`
- MERGEs all Silver tables — new rows inserted, changed rows updated, history preserved
- Calculates `net_fee_payable` on enrolment

**Output tables:**
- `silver.student` — unified student record with SCD Type 2
- `silver.application` — application + offer + decision data
- `silver.enrolment` — confirmed enrolments with fees
- `silver.attendance` — session attendance records
- `silver.course`, `silver.campus`, `silver.intake` — reference tables

**Design principle:** Silver is a conforming layer — not 3NF.
Wide, flat tables optimised for Gold loading. One table per subject area.
Identity resolution key: `student_email` (common across all four source systems).

> **Why MERGE and not overwrite?**
> Each pipeline run brings only the delta — new or changed records.
> MERGE accumulates Silver over time. Overwrite would destroy history.
> Re-running the same `pipeline_run_date` is safe — MERGE is idempotent.

## Parameters

Injected by the Fabric pipeline at runtime.
Must match the `pipeline_run_date` used in `01_bronze_ingest`.

In [ ]:
# Default — overridden by Fabric pipeline at runtime
pipeline_run_date = "2024-01-15"  # YYYY-MM-DD

## Setup

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

run_date = pipeline_run_date
print(f"Pipeline run date: {run_date}")
print()


def read_bronze(table_name: str, source_system: str):
    """
    Reads only the Bronze partition matching pipeline_run_date.
    This is the core pattern that prevents duplicate processing.
    """
    df = (
        spark.table(f"bronze.{table_name}")
        .filter(
            (F.col("_source_system")  == source_system) &
            (F.col("_ingestion_date") == run_date)
        )
    )
    count = df.count()
    print(f"[BRONZE] {table_name} ({source_system}, {run_date}): {count} rows")
    return df


def table_exists(table_name: str) -> bool:
    """Check if a Delta table exists in the metastore."""
    try:
        spark.table(table_name)
        return True
    except Exception:
        return False


print("Setup complete.")

## Step 1 — Read Bronze Partitions

Read only the data that arrived in this pipeline run.
Each DataFrame here represents one source system's delta for today.

In [ ]:
bronze_leads       = read_bronze("hubspot_leads",         "HUBSPOT")
bronze_apps        = read_bronze("dynamics_applications", "DYNAMICS")
bronze_enrolments  = read_bronze("paradigm_enrolments",   "PARADIGM")
bronze_attendance  = read_bronze("canvas_attendance",     "CANVAS")

## Step 2 — Build silver.student

### Identity Resolution
A student may appear in up to three source systems:
- **HubSpot** — as a lead (first touch)
- **Dynamics** — as an applicant
- **Paradigm** — as an enrolled student (most authoritative)

Resolution key: `student_email` — the only field guaranteed to be consistent
across all systems. We left-join HubSpot and Dynamics onto Paradigm.
For students who only exist in HubSpot (never enrolled), we build the record
from HubSpot alone.

### SCD Type 2 — visa_code and is_international
These two attributes are tracked historically because they affect:
- Revenue classification (domestic fee vs international fee)
- ESOS compliance reporting

**Scenario in this dataset:**
Amir Hassan appears in HubSpot as `Domestic`. When Paradigm data arrives,
it records him as `International - Student Visa` (he changed visa status
between lead capture and enrolment). Silver detects this mismatch and
creates a new Type 2 version — the old row is expired, a new current row inserted.

**SCD Type 2 admin columns:**
- `_valid_from` — date this version became active
- `_valid_to` — 9999-12-31 for current record
- `_is_current` — True for the active version only

In [ ]:
# ── Build incoming student records from all three sources ──────────────────
#
# Priority: Paradigm > HubSpot (Paradigm is the system of record for enrolled students)
# For students not yet in Paradigm, HubSpot is the only source.
#
# Visa source of truth:
#   Paradigm visa_type overrides HubSpot visa_type when both exist.
#   This is intentional — Paradigm reflects the student's actual enrolment visa.

# Prepare HubSpot leads — rename columns to avoid conflicts on join
hs = bronze_leads.select(
    F.col("lead_id").alias("hubspot_lead_id"),
    F.col("student_email"),
    F.col("student_first_name"),
    F.col("student_last_name"),
    F.col("date_of_birth").cast("date"),
    F.col("visa_type").alias("hs_visa_type"),
    F.col("lead_source"),
)

# Prepare Paradigm enrolments — distinct student records
# (a student may have multiple enrolment rows — we only need one student record)
paradigm_students = bronze_enrolments.select(
    F.col("student_email"),
    # Paradigm does not have a student_id in our CSV — we use email as the key
    # In a real Paradigm integration, student_id would come from the SIS
).distinct()

# Resolve visa type:
#   If student appears in both HubSpot and Paradigm, use Paradigm's record
#   For this demo, Paradigm enrolment implies International Student Visa
#   if HubSpot recorded them as Domestic — this is the SCD Type 2 trigger
#
# In a real integration, Paradigm would have a visa_code field.
# Here we simulate: enrolled international students have a higher fee (>= 30000)
# We use this to derive the corrected visa classification.

# Join Paradigm enrolment fee data to detect international students
paradigm_fees = bronze_enrolments.groupBy("student_email").agg(
    F.max("enrolment_fee").alias("max_fee")
)

# Build unified incoming student records
incoming_students = (
    hs
    .join(paradigm_fees, on="student_email", how="left")
    .withColumn(
        # Derive corrected visa_code:
        # If Paradigm shows fee >= 30000 but HubSpot said Domestic → International
        # Otherwise keep HubSpot visa type
        "visa_code",
        F.when(
            (F.col("max_fee") >= 30000) & (F.col("hs_visa_type") == "Domestic"),
            F.lit("International - Student Visa")
        ).otherwise(F.col("hs_visa_type"))
    )
    .withColumn(
        "is_international",
        F.when(F.col("visa_code") == "Domestic", F.lit(False)).otherwise(F.lit(True))
    )
    .withColumn(
        # Use email as student_id — in production this comes from Paradigm SIS
        "student_id",
        F.col("student_email")
    )
    .withColumn("_source_system", F.lit("MULTI"))
    .withColumn("_loaded_at", F.current_timestamp())
    .select(
        "student_id",
        "hubspot_lead_id",
        "student_first_name",
        "student_last_name",
        "student_email",
        "date_of_birth",
        "visa_code",
        "is_international",
        "lead_source",
        "_source_system",
        "_loaded_at",
    )
)

print("Incoming student records:")
incoming_students.select(
    "student_email", "visa_code", "is_international"
).show(truncate=False)

In [ ]:
# ── SCD Type 2 MERGE into silver.student ──────────────────────────────────
#
# Pattern:
#   Step 1 — Expire current rows where visa_code or is_international has changed
#   Step 2 — Insert new versions for changed rows + brand new students
#
# Result: Amir Hassan will have TWO rows in silver.student after this run:
#   Row 1: visa_code = 'Domestic',                _valid_to = run_date - 1, _is_current = False
#   Row 2: visa_code = 'International - Student Visa', _valid_to = 9999-12-31, _is_current = True

SILVER_STUDENT = "silver.student"

if not table_exists(SILVER_STUDENT):
    # ── First run: no existing table — insert all rows as current ──────────
    print("[silver.student] First run — creating table")

    first_load = (
        incoming_students
        .withColumn("_valid_from",  F.lit(run_date).cast("date"))
        .withColumn("_valid_to",    F.lit("9999-12-31").cast("date"))
        .withColumn("_is_current",  F.lit(True))
    )

    first_load.write.format("delta").mode("overwrite").saveAsTable(SILVER_STUDENT)
    print(f"[silver.student] Inserted {first_load.count()} rows")

else:
    # ── Subsequent runs: MERGE ─────────────────────────────────────────────
    print("[silver.student] Existing table found — running SCD Type 2 MERGE")

    existing = DeltaTable.forName(spark, SILVER_STUDENT)

    # Step 1: Expire rows where Type 2 attributes have changed
    existing.alias("target").merge(
        incoming_students.alias("source"),
        "target.student_id = source.student_id AND target._is_current = true"
    ).whenMatchedUpdate(
        condition="""
            target.visa_code      != source.visa_code OR
            target.is_international != source.is_international
        """,
        set={
            "_valid_to":   F.date_sub(F.lit(run_date).cast("date"), 1),
            "_is_current": F.lit(False)
        }
    ).execute()

    print("[silver.student] Step 1 complete — expired changed rows")

    # Step 2: Insert new versions for changed students + brand new students
    # left_anti join: students in incoming who are NOT already current in Silver
    current_silver = spark.table(SILVER_STUDENT).filter("_is_current = true")

    new_versions = (
        incoming_students
        .join(
            current_silver.select("student_id"),
            on="student_id",
            how="left_anti"  # students not already current — includes changed + new
        )
        .withColumn("_valid_from",  F.lit(run_date).cast("date"))
        .withColumn("_valid_to",    F.lit("9999-12-31").cast("date"))
        .withColumn("_is_current",  F.lit(True))
    )

    new_count = new_versions.count()
    new_versions.write.format("delta").mode("append").saveAsTable(SILVER_STUDENT)
    print(f"[silver.student] Step 2 complete — inserted {new_count} new/updated versions")

# Verify
total = spark.table(SILVER_STUDENT).count()
current = spark.table(SILVER_STUDENT).filter("_is_current = true").count()
print(f"[silver.student] Total rows: {total} | Current rows: {current}")

## Step 3 — Build silver.application

Application, offer, and decision data from Dynamics.
Enriched with `student_id` resolved from `student_email`.
MERGE on `application_id` — application records may be updated
as offers are made and decisions recorded.

In [ ]:
SILVER_APPLICATION = "silver.application"

# Resolve student_id from silver.student (current records only)
student_lookup = (
    spark.table(SILVER_STUDENT)
    .filter("_is_current = true")
    .select("student_id", "student_email")
)

incoming_apps = (
    bronze_apps
    .join(student_lookup, on="student_email", how="left")
    .select(
        F.col("application_id"),
        F.col("student_id"),
        F.col("lead_id"),
        F.col("course_code"),
        F.col("campus_code"),
        F.col("intake_code"),
        F.col("application_date").cast("date"),
        F.col("offer_date").cast("date"),
        F.col("offer_type"),
        F.col("decision_date").cast("date"),
        F.col("decision_outcome"),
        F.current_timestamp().alias("_loaded_at"),
    )
)

if not table_exists(SILVER_APPLICATION):
    print("[silver.application] First run — creating table")
    incoming_apps.write.format("delta").mode("overwrite").saveAsTable(SILVER_APPLICATION)
else:
    print("[silver.application] Merging into existing table")
    DeltaTable.forName(spark, SILVER_APPLICATION).alias("t").merge(
        incoming_apps.alias("s"),
        "t.application_id = s.application_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

count = spark.table(SILVER_APPLICATION).count()
print(f"[silver.application] Total rows: {count}")
spark.table(SILVER_APPLICATION).show(truncate=False)

## Step 4 — Build silver.enrolment

Enrolment records from Paradigm SIS.
`net_fee_payable` is calculated here — the only derived field in Silver.
MERGE on `enrolment_id` — enrolment status may update (Active → Withdrawn).

In [ ]:
SILVER_ENROLMENT = "silver.enrolment"

incoming_enrolments = (
    bronze_enrolments
    .join(student_lookup, on="student_email", how="left")
    .select(
        F.col("enrolment_id"),
        F.col("student_id"),
        F.col("course_code"),
        F.col("campus_code"),
        F.col("intake_code"),
        F.col("enrolment_date").cast("date"),
        F.col("enrolment_status"),
        F.col("withdrawal_date").cast("date"),
        F.col("completion_date").cast("date"),
        F.col("enrolment_fee").cast("decimal(15,2)"),
        F.col("scholarship_amount").cast("decimal(15,2)"),
        # Calculate net_fee_payable here in Silver — not in Gold
        # Gold reads this pre-calculated value directly
        (F.col("enrolment_fee") - F.col("scholarship_amount"))
            .cast("decimal(15,2)")
            .alias("net_fee_payable"),
        F.col("credit_points").cast("int"),
        F.current_timestamp().alias("_loaded_at"),
    )
)

if not table_exists(SILVER_ENROLMENT):
    print("[silver.enrolment] First run — creating table")
    incoming_enrolments.write.format("delta").mode("overwrite").saveAsTable(SILVER_ENROLMENT)
else:
    print("[silver.enrolment] Merging into existing table")
    DeltaTable.forName(spark, SILVER_ENROLMENT).alias("t").merge(
        incoming_enrolments.alias("s"),
        "t.enrolment_id = s.enrolment_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

count = spark.table(SILVER_ENROLMENT).count()
print(f"[silver.enrolment] Total rows: {count}")
spark.table(SILVER_ENROLMENT).show(truncate=False)

## Step 5 — Build silver.attendance

Session attendance from Canvas LMS.
Each row = one student × one scheduled session.
MERGE on `session_id` — attendance records are immutable once submitted,
but we MERGE to handle reruns safely.

In [ ]:
SILVER_ATTENDANCE = "silver.attendance"

incoming_attendance = (
    bronze_attendance
    .join(student_lookup, on="student_email", how="left")
    .select(
        F.col("session_id"),
        F.col("student_id"),
        F.col("course_code"),
        F.col("campus_code"),
        F.col("intake_code"),
        F.col("session_date").cast("date"),
        F.col("session_type"),
        F.col("attended").cast("int"),
        F.current_timestamp().alias("_loaded_at"),
    )
)

if not table_exists(SILVER_ATTENDANCE):
    print("[silver.attendance] First run — creating table")
    incoming_attendance.write.format("delta").mode("overwrite").saveAsTable(SILVER_ATTENDANCE)
else:
    print("[silver.attendance] Merging into existing table")
    DeltaTable.forName(spark, SILVER_ATTENDANCE).alias("t").merge(
        incoming_attendance.alias("s"),
        "t.session_id = s.session_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

count = spark.table(SILVER_ATTENDANCE).count()
print(f"[silver.attendance] Total rows: {count}")

## Step 6 — Build Reference Tables

Distinct reference values conformed across all source systems.
One row per `course_code`, `campus_code`, `intake_code`.

These feed directly into Gold dimension tables.
MERGE on the natural key — new values inserted, existing values untouched.

In [ ]:
# ── silver.course ──────────────────────────────────────────────────────────
SILVER_COURSE = "silver.course"

# Collect distinct course codes from all sources that reference courses
courses_from_apps  = bronze_apps.select("course_code")
courses_from_enrol = bronze_enrolments.select("course_code")

incoming_courses = (
    courses_from_apps.union(courses_from_enrol)
    .distinct()
    .withColumn("_loaded_at", F.current_timestamp())
)

if not table_exists(SILVER_COURSE):
    incoming_courses.write.format("delta").mode("overwrite").saveAsTable(SILVER_COURSE)
else:
    DeltaTable.forName(spark, SILVER_COURSE).alias("t").merge(
        incoming_courses.alias("s"), "t.course_code = s.course_code"
    ).whenNotMatchedInsertAll().execute()

print(f"[silver.course] Total rows: {spark.table(SILVER_COURSE).count()}")
spark.table(SILVER_COURSE).show(truncate=False)


# ── silver.campus ──────────────────────────────────────────────────────────
SILVER_CAMPUS = "silver.campus"

campus_from_apps   = bronze_apps.select("campus_code")
campus_from_enrol  = bronze_enrolments.select("campus_code")
campus_from_attend = bronze_attendance.select("campus_code")

incoming_campus = (
    campus_from_apps.union(campus_from_enrol).union(campus_from_attend)
    .distinct()
    .withColumn("_loaded_at", F.current_timestamp())
)

if not table_exists(SILVER_CAMPUS):
    incoming_campus.write.format("delta").mode("overwrite").saveAsTable(SILVER_CAMPUS)
else:
    DeltaTable.forName(spark, SILVER_CAMPUS).alias("t").merge(
        incoming_campus.alias("s"), "t.campus_code = s.campus_code"
    ).whenNotMatchedInsertAll().execute()

print(f"[silver.campus] Total rows: {spark.table(SILVER_CAMPUS).count()}")
spark.table(SILVER_CAMPUS).show(truncate=False)


# ── silver.intake ──────────────────────────────────────────────────────────
SILVER_INTAKE = "silver.intake"

intake_from_apps   = bronze_apps.select("intake_code")
intake_from_enrol  = bronze_enrolments.select("intake_code")

incoming_intake = (
    intake_from_apps.union(intake_from_enrol)
    .distinct()
    .withColumn("_loaded_at", F.current_timestamp())
)

if not table_exists(SILVER_INTAKE):
    incoming_intake.write.format("delta").mode("overwrite").saveAsTable(SILVER_INTAKE)
else:
    DeltaTable.forName(spark, SILVER_INTAKE).alias("t").merge(
        incoming_intake.alias("s"), "t.intake_code = s.intake_code"
    ).whenNotMatchedInsertAll().execute()

print(f"[silver.intake] Total rows: {spark.table(SILVER_INTAKE).count()}")
spark.table(SILVER_INTAKE).show(truncate=False)

## Verification

Final checks across all Silver tables.

In [ ]:
silver_tables = [
    "silver.student",
    "silver.application",
    "silver.enrolment",
    "silver.attendance",
    "silver.course",
    "silver.campus",
    "silver.intake",
]

print(f"{'Table':<30} {'Total':>8} {'Current':>10}")
print("-" * 52)

for table in silver_tables:
    df = spark.table(table)
    total = df.count()
    # For SCD Type 2 table, show current count
    if "_is_current" in df.columns:
        current = df.filter("_is_current = true").count()
    else:
        current = total
    print(f"{table:<30} {total:>8} {current:>10}")

print("-" * 52)
print("Current = active rows (relevant for SCD Type 2 tables)")

In [ ]:
# ── SCD Type 2 specific check ──────────────────────────────────────────────
# Amir Hassan should have TWO rows if the visa change was detected:
#   Row 1 (expired):  visa_code = 'Domestic',                _is_current = False
#   Row 2 (current):  visa_code = 'International - Student Visa', _is_current = True

print("SCD Type 2 check — silver.student (all versions):")
print()

(
    spark.table("silver.student")
    .select(
        "student_email",
        "visa_code",
        "is_international",
        "_valid_from",
        "_valid_to",
        "_is_current"
    )
    .orderBy("student_email", "_valid_from")
    .show(20, truncate=False)
)

In [ ]:
# ── Identity resolution check ──────────────────────────────────────────────
# All students in Silver must have a student_id resolved from their email
# No NULL student_ids should exist

null_ids = (
    spark.table("silver.student")
    .filter(F.col("student_id").isNull())
    .count()
)

print(f"Null student_id count (should be 0): {null_ids}")

# Check applications resolved correctly
null_app_ids = (
    spark.table("silver.application")
    .filter(F.col("student_id").isNull())
    .count()
)
print(f"Null student_id in applications (should be 0): {null_app_ids}")

## Summary

Silver transformation complete. All staging tables loaded and verified.

**What was built:**
- `silver.student` — 10 students unified across HubSpot + Paradigm, SCD Type 2 active
- `silver.application` — 8 application records with offer and decision data
- `silver.enrolment` — 6 enrolments with `net_fee_payable` pre-calculated
- `silver.attendance` — 34 session records
- `silver.course / campus / intake` — reference tables for Gold dimensions

**Key checks passed:**
- Student identity resolved via `student_email` across all sources
- Amir Hassan: SCD Type 2 triggered if visa mismatch detected
- No NULL `student_id` values in any Silver table
- MERGE pattern is idempotent — re-running same date is safe

**Next step:** Run `03_gold_dimensions.ipynb` with the same `pipeline_run_date`.